# 01. Исследовательский анализ данных и генерация доменных признаков (EDA & Feature Engineering)

В данном ноутбуке представлен воспроизводимый исследовательский анализ факторов оттока сотрудников, проверка статистических взаимосвязей и создание доменных HR-индексов без утечек данных.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import DATA_PROCESSED_DIR, TARGET_COL
from src.features import HRFeatureTransformer, compute_mutual_information

## 1. Загрузка подготовленной обучающей выборки (Train-Dev Set)

In [ ]:
df_train = pd.read_parquet(DATA_PROCESSED_DIR / "train_dev.parquet")
print("Размерность обучающей выборки:", df_train.shape)
print("Доля уволившихся сотрудников:", f"{df_train[TARGET_COL].mean():.2%}")
df_train.head()

## 2. Анализ ключевых факторов риска: Сверхурочная работа и доход

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.barplot(data=df_train, x="OverTime", y=TARGET_COL, errorbar=None, palette="Blues_d")
plt.title("Доля оттока в зависимости от OverTime")
plt.ylabel("Вероятность увольнения")

plt.subplot(1, 2, 2)
sns.boxplot(data=df_train, x=TARGET_COL, y="MonthlyIncome", palette="Set2")
plt.title("Распределение MonthlyIncome по классам")
plt.tight_layout()
plt.show()

## 3. Применение трансформера признаков (HRFeatureTransformer)

Трансформер рассчитывает медианы ролей строго на обучающих данных и формирует доменные индексы:
- Promotion_Stagnation_Ratio
- Income_to_Role_Median
- Role_Loyalty_Ratio
- Tenure_Per_Company
- Total_Satisfaction_Index
- Burnout_Risk_Score
- Training_Investment_Ratio
- Commute_Distance_Ratio
- Tenure_to_Age

In [ ]:
transformer = HRFeatureTransformer()
X_train = df_train.drop(columns=[TARGET_COL, "fold"])
y_train = df_train[TARGET_COL]
X_engineered = transformer.fit_transform(X_train, y_train)
print("Признаковое пространство расширено до:", X_engineered.shape[1], "колонок")
X_engineered[["Promotion_Stagnation_Ratio", "Income_to_Role_Median", "Burnout_Risk_Score"]].head()

## 4. Оценка взаимной информации (Mutual Information) новых признаков

In [ ]:
mi_scores = compute_mutual_information(X_engineered, y_train)
plt.figure(figsize=(10, 6))
mi_scores.head(15).plot(kind="barh", color="teal")
plt.title("Топ-15 признаков по взаимной информации с оттоком")
plt.xlabel("Mutual Information")
plt.gca().invert_yaxis()
plt.show()